In [18]:
import pandas as pd
import numpy as np
import joblib
import json
import os
import time

from sklearn.model_selection import train_test_split, RandomizedSearchCV, KFold, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error
from xgboost import XGBRegressor

# Folder output -> dipakai oleh app.py, JANGAN diubah namanya
OUTPUT_DIR = 'outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [19]:
kolom_dipakai = ['price', 'year', 'manufacturer', 'condition', 'odometer',
                 'cylinders', 'fuel', 'transmission']

df_raw = pd.read_csv('data/vehicles.csv', usecols=kolom_dipakai)

print(f"Jumlah baris awal (sebelum cleaning): {len(df_raw):,}")
df_raw.head()

Jumlah baris awal (sebelum cleaning): 426,880


,price,year,manufacturer,condition,cylinders,fuel,odometer,transmission
0,6000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,11900,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,21000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1500,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4900,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [20]:
df = df_raw.copy()

# Cek persentase missing value per kolom
missing_pct = (df.isna().sum() / len(df) * 100).sort_values(ascending=False)
print("Persentase missing value per kolom:")
print(missing_pct)

# Drop kolom dengan missing > 40% (KECUALI fitur wajib sesuai proposal:
# condition & cylinders tetap dipertahankan meski missing-nya tinggi,
# nanti diisi modus, bukan dihapus kolomnya)
fitur_wajib = ['price', 'year', 'manufacturer', 'condition', 'odometer',
               'cylinders', 'fuel', 'transmission']
kolom_dihapus = [c for c in missing_pct[missing_pct > 40].index.tolist() if c not in fitur_wajib]
if kolom_dihapus:
    print(f"\nKolom dihapus (missing > 40%): {kolom_dihapus}")
    df = df.drop(columns=kolom_dihapus)
else:
    print("\nTidak ada kolom yang dihapus (semua fitur wajib dipertahankan).")

# price wajib ada -> baris tanpa price dibuang (tidak ada target = tidak bisa dipakai untuk supervised learning)
df = df.dropna(subset=['price'])

# Kolom numerik -> median
kolom_numerik = [c for c in ['year', 'odometer'] if c in df.columns]
for col in kolom_numerik:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)
    print(f"Kolom '{col}' diisi median = {median_val}")

# Kolom kategorikal -> modus
kolom_kategorikal = [c for c in ['manufacturer', 'condition', 'cylinders', 'fuel', 'transmission'] if c in df.columns]
for col in kolom_kategorikal:
    modus_val = df[col].mode()[0]
    df[col] = df[col].fillna(modus_val)
    print(f"Kolom '{col}' diisi modus = {modus_val}")

print(f"\nJumlah baris setelah handling missing values: {len(df):,}")

Persentase missing value per kolom:
cylinders       41.622470
condition       40.785232
manufacturer     4.133714
odometer         1.030735
fuel             0.705819
transmission     0.598763
year             0.282281
price            0.000000
dtype: float64

Tidak ada kolom yang dihapus (semua fitur wajib dipertahankan).
Kolom 'year' diisi median = 2013.0
Kolom 'odometer' diisi median = 85548.0
Kolom 'manufacturer' diisi modus = ford
Kolom 'condition' diisi modus = good
Kolom 'cylinders' diisi modus = 6 cylinders
Kolom 'fuel' diisi modus = gas
Kolom 'transmission' diisi modus = automatic

Jumlah baris setelah handling missing values: 426,880


In [21]:
print(df.columns.tolist())

['price', 'year', 'manufacturer', 'condition', 'cylinders', 'fuel', 'odometer', 'transmission']


In [22]:
brands_app = ['toyota', 'honda', 'suzuki', 'nissan', 'ford']
conditions_app = ['excellent', 'good', 'fair', 'poor']

df['manufacturer'] = df['manufacturer'].astype(str).str.lower().str.strip()
df['condition'] = df['condition'].astype(str).str.lower().str.strip()

df = df[df['manufacturer'].isin(brands_app)]
df = df[df['condition'].isin(conditions_app)]

print(f"Jumlah baris setelah filter brand & condition: {len(df):,}")

Jumlah baris setelah filter brand & condition: 154,051


In [23]:
# Filter harga & tahun (rule-based, sesuai proposal)
df = df[(df['price'] > 500) & (df['price'] < 100000)]
df = df[(df['year'] >= 2000) & (df['year'] <= 2026)]
df = df[(df['odometer'] > 100) & (df['odometer'] < 500000)]

print(f"Jumlah baris setelah filter rule-based: {len(df):,}")

# IQR untuk odometer
Q1 = df['odometer'].quantile(0.25)
Q3 = df['odometer'].quantile(0.75)
IQR = Q3 - Q1
batas_bawah = Q1 - 1.5 * IQR
batas_atas = Q3 + 1.5 * IQR

print(f"Q1={Q1}, Q3={Q3}, IQR={IQR}")
print(f"Batas IQR odometer: [{batas_bawah:.0f}, {batas_atas:.0f}]")

df = df[(df['odometer'] >= max(batas_bawah, 0)) & (df['odometer'] <= batas_atas)]

print(f"Jumlah baris FINAL setelah IQR outlier removal: {len(df):,}")

Jumlah baris setelah filter rule-based: 125,980
Q1=48315.0, Q3=147989.75, IQR=99674.75
Batas IQR odometer: [-101197, 297502]
Jumlah baris FINAL setelah IQR outlier removal: 125,271


In [24]:
TAHUN_SEKARANG = 2026
df['car_age'] = TAHUN_SEKARANG - df['year']
df['car_age'] = df['car_age'].clip(lower=0)
df[['year', 'car_age']].describe()

,year,car_age
count,125271.000000,125271.000000
mean,2012.261609,13.738391
std,5.100854,5.100854
min,2000.000000,4.000000
25%,2008.000000,9.000000
50%,2013.000000,13.000000
75%,2017.000000,18.000000
max,2022.000000,26.000000


In [25]:
brand_map = {'toyota': 0, 'honda': 1, 'suzuki': 2, 'nissan': 3, 'ford': 4}
cond_map = {'excellent': 0, 'good': 1, 'fair': 2, 'poor': 3}

df['brand_encoded'] = df['manufacturer'].map(brand_map)
df['cond_encoded'] = df['condition'].map(cond_map)

# One-hot encoding untuk cylinders, fuel, transmission
kolom_onehot = [c for c in ['cylinders', 'fuel', 'transmission'] if c in df.columns]
df_encoded = pd.get_dummies(df, columns=kolom_onehot, prefix=kolom_onehot)

print("Kolom hasil encoding:")
print(df_encoded.columns.tolist())

# Simpan daftar kategori unik untuk cylinders/fuel/transmission -> dipakai di app.py untuk dropdown
kategori_dropdown = {}
for col in kolom_onehot:
    kategori_dropdown[col] = sorted(df[col].dropna().unique().tolist())

print("\nKategori unik untuk dropdown app.py:")
print(kategori_dropdown)

with open(f'{OUTPUT_DIR}/kategori_dropdown.json', 'w') as f:
    json.dump(kategori_dropdown, f, indent=2)

Kolom hasil encoding:
['price', 'year', 'manufacturer', 'condition', 'odometer', 'car_age', 'brand_encoded', 'cond_encoded', 'cylinders_10 cylinders', 'cylinders_12 cylinders', 'cylinders_3 cylinders', 'cylinders_4 cylinders', 'cylinders_5 cylinders', 'cylinders_6 cylinders', 'cylinders_8 cylinders', 'cylinders_other', 'fuel_diesel', 'fuel_electric', 'fuel_gas', 'fuel_hybrid', 'fuel_other', 'transmission_automatic', 'transmission_manual', 'transmission_other']

Kategori unik untuk dropdown app.py:
{'cylinders': ['10 cylinders', '12 cylinders', '3 cylinders', '4 cylinders', '5 cylinders', '6 cylinders', '8 cylinders', 'other'], 'fuel': ['diesel', 'electric', 'gas', 'hybrid', 'other'], 'transmission': ['automatic', 'manual', 'other']}


In [26]:
fitur_dasar = ['brand_encoded', 'year', 'odometer', 'cond_encoded', 'car_age']
fitur_onehot = [c for c in df_encoded.columns if any(c.startswith(p + '_') for p in kolom_onehot)]
fitur_final = fitur_dasar + fitur_onehot

print(f"Total fitur yang dipakai model ({len(fitur_final)}):")
print(fitur_final)

X = df_encoded[fitur_final].copy()
y = df_encoded['price'].copy()

# Simpan daftar nama kolom fitur -> WAJIB dipakai app.py supaya urutan kolom konsisten saat predict
with open(f'{OUTPUT_DIR}/feature_columns.json', 'w') as f:
    json.dump(fitur_final, f, indent=2)

X.head()

Total fitur yang dipakai model (21):
['brand_encoded', 'year', 'odometer', 'cond_encoded', 'car_age', 'cylinders_10 cylinders', 'cylinders_12 cylinders', 'cylinders_3 cylinders', 'cylinders_4 cylinders', 'cylinders_5 cylinders', 'cylinders_6 cylinders', 'cylinders_8 cylinders', 'cylinders_other', 'fuel_diesel', 'fuel_electric', 'fuel_gas', 'fuel_hybrid', 'fuel_other', 'transmission_automatic', 'transmission_manual', 'transmission_other']


,brand_encoded,year,odometer,cond_encoded,car_age,cylinders_10 cylinders,cylinders_12 cylinders,cylinders_3 cylinders,cylinders_4 cylinders,cylinders_5 cylinders,...,cylinders_8 cylinders,cylinders_other,fuel_diesel,fuel_electric,fuel_gas,fuel_hybrid,fuel_other,transmission_automatic,transmission_manual,transmission_other
0,4,2013.0,85548.0,1,13.0,False,False,False,False,False,...,False,False,False,False,True,False,False,True,False,False
1,4,2013.0,85548.0,1,13.0,False,False,False,False,False,...,False,False,False,False,True,False,False,True,False,False
2,4,2013.0,85548.0,1,13.0,False,False,False,False,False,...,False,False,False,False,True,False,False,True,False,False
3,4,2013.0,85548.0,1,13.0,False,False,False,False,False,...,False,False,False,False,True,False,False,True,False,False
4,4,2013.0,85548.0,1,13.0,False,False,False,False,False,...,False,False,False,False,True,False,False,True,False,False


In [27]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

print(f"Jumlah data training: {len(X_train):,}")
print(f"Jumlah data testing : {len(X_test):,}")

Jumlah data training: 100,216
Jumlah data testing : 25,055


In [28]:
def evaluasi_model(nama, model, X_test, y_test, y_pred, waktu_training):
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    mape = mean_absolute_percentage_error(y_test, y_pred) * 100
    print(f"[{nama}] MAE={mae:.2f} | RMSE={rmse:.2f} | R2={r2:.4f} | MAPE={mape:.2f}% | waktu={waktu_training:.2f}s")
    return {
        'model': nama,
        'MAE': round(mae, 2),
        'RMSE': round(rmse, 2),
        'R2': round(r2, 4),
        'MAPE': round(mape, 2),
        'training_time_sec': round(waktu_training, 2)
    }

hasil_komparasi = []
trained_models = {}

In [29]:
# --- 1. Linear Regression ---
pipe_lr = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LinearRegression())
])

t0 = time.time()
pipe_lr.fit(X_train, y_train)
waktu_lr = time.time() - t0

pred_lr = pipe_lr.predict(X_test)
hasil_komparasi.append(evaluasi_model('Linear Regression', pipe_lr, X_test, y_test, pred_lr, waktu_lr))
trained_models['Linear Regression'] = pipe_lr

[Linear Regression] MAE=6436.02 | RMSE=8893.07 | R2=0.6084 | MAPE=78.85% | waktu=0.22s


In [30]:
# --- 2. KNN Regressor ---
pipe_knn = Pipeline([
    ('scaler', StandardScaler()),
    ('model', KNeighborsRegressor(n_neighbors=7))
])

t0 = time.time()
pipe_knn.fit(X_train, y_train)
waktu_knn = time.time() - t0

pred_knn = pipe_knn.predict(X_test)
hasil_komparasi.append(evaluasi_model('KNN Regressor', pipe_knn, X_test, y_test, pred_knn, waktu_knn))
trained_models['KNN Regressor'] = pipe_knn

[KNN Regressor] MAE=3609.96 | RMSE=6420.05 | R2=0.7959 | MAPE=49.62% | waktu=0.17s


In [31]:
# --- 3. Random Forest ---
model_rf = RandomForestRegressor(
    n_estimators=200, max_depth=12, random_state=RANDOM_STATE, n_jobs=-1
)

t0 = time.time()
model_rf.fit(X_train, y_train)
waktu_rf = time.time() - t0

pred_rf = model_rf.predict(X_test)
hasil_komparasi.append(evaluasi_model('Random Forest', model_rf, X_test, y_test, pred_rf, waktu_rf))
trained_models['Random Forest'] = model_rf

[Random Forest] MAE=4302.75 | RMSE=6679.84 | R2=0.7790 | MAPE=56.09% | waktu=8.01s


In [32]:
param_dist = {
    'n_estimators': [100, 200, 300, 400, 500],
    'max_depth': [3, 4, 5, 6, 7, 8],
    'learning_rate': [0.01, 0.05, 0.1, 0.15, 0.2, 0.3],
    'subsample': [0.6, 0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0],
}

xgb_base = XGBRegressor(random_state=RANDOM_STATE, n_jobs=-1)

random_search = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=param_dist,
    n_iter=25,
    scoring='neg_mean_absolute_error',
    cv=5,  # 5-Fold Cross Validation
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1
)

t0 = time.time()
random_search.fit(X_train, y_train)
waktu_xgb = time.time() - t0

print(f"\nParameter terbaik XGBoost: {random_search.best_params_}")
print(f"Waktu tuning + training: {waktu_xgb:.2f}s")

model_xgb = random_search.best_estimator_

Fitting 5 folds for each of 25 candidates, totalling 125 fits

Parameter terbaik XGBoost: {'subsample': 1.0, 'n_estimators': 400, 'max_depth': 8, 'learning_rate': 0.2, 'colsample_bytree': 0.9}
Waktu tuning + training: 77.24s


In [33]:
# 5-Fold CV score tambahan untuk model XGBoost terbaik (laporan generalisasi)
kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_scores = cross_val_score(model_xgb, X_train, y_train, cv=kf, scoring='r2')
print(f"5-Fold CV R2 scores: {cv_scores}")
print(f"Rata-rata CV R2: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

pred_xgb = model_xgb.predict(X_test)
hasil_komparasi.append(evaluasi_model('XGBoost', model_xgb, X_test, y_test, pred_xgb, waktu_xgb))
trained_models['XGBoost'] = model_xgb

5-Fold CV R2 scores: [0.79283482 0.80199492 0.80217564 0.7974661  0.79553652]
Rata-rata CV R2: 0.7980 (+/- 0.0036)
[XGBoost] MAE=3769.46 | RMSE=6195.31 | R2=0.8099 | MAPE=50.00% | waktu=77.24s


In [34]:
df_komparasi = pd.DataFrame(hasil_komparasi)
df_komparasi = df_komparasi.sort_values('R2', ascending=False).reset_index(drop=True)
df_komparasi['Rank'] = df_komparasi.index + 1
print(df_komparasi.to_string(index=False))

df_komparasi.to_csv(f'{OUTPUT_DIR}/model_comparison.csv', index=False)
print(f"\nTersimpan ke {OUTPUT_DIR}/model_comparison.csv")

            model     MAE    RMSE     R2  MAPE  training_time_sec  Rank
          XGBoost 3769.46 6195.31 0.8099 50.00              77.24     1
    KNN Regressor 3609.96 6420.05 0.7959 49.62               0.17     2
    Random Forest 4302.75 6679.84 0.7790 56.09               8.01     3
Linear Regression 6436.02 8893.07 0.6084 78.85               0.22     4

Tersimpan ke outputs/model_comparison.csv


In [35]:
nama_model_terbaik = df_komparasi.iloc[0]['model']
model_final = trained_models[nama_model_terbaik]

print(f"Model final terpilih: {nama_model_terbaik}")

joblib.dump(model_final, 'carprice_xgb.pkl')
joblib.dump(model_final, f'{OUTPUT_DIR}/model_final.pkl')

with open(f'{OUTPUT_DIR}/model_info.json', 'w') as f:
    json.dump({
        'nama_model_terbaik': nama_model_terbaik,
        'jumlah_data_final': len(df),
        'jumlah_fitur': len(fitur_final),
        'cv_r2_mean': round(cv_scores.mean(), 4),
        'cv_r2_std': round(cv_scores.std(), 4),
        'best_xgb_params': random_search.best_params_,
    }, f, indent=2)

print("Model & info tersimpan.")

Model final terpilih: XGBoost
Model & info tersimpan.


In [36]:
importances = model_xgb.feature_importances_
df_importance = pd.DataFrame({
    'fitur': fitur_final,
    'importance': importances
}).sort_values('importance', ascending=False).reset_index(drop=True)

print(df_importance.to_string(index=False))
df_importance.to_csv(f'{OUTPUT_DIR}/feature_importance.csv', index=False)

                 fitur  importance
           fuel_diesel    0.282258
cylinders_12 cylinders    0.118764
                  year    0.087842
 cylinders_4 cylinders    0.067750
 cylinders_8 cylinders    0.053442
 cylinders_6 cylinders    0.044640
    transmission_other    0.043116
 cylinders_3 cylinders    0.037831
cylinders_10 cylinders    0.035829
            fuel_other    0.035334
              odometer    0.032616
           fuel_hybrid    0.029066
         fuel_electric    0.025936
         brand_encoded    0.022585
              fuel_gas    0.014098
 cylinders_5 cylinders    0.013998
transmission_automatic    0.013522
       cylinders_other    0.011749
               car_age    0.011057
          cond_encoded    0.009752
   transmission_manual    0.008815


In [37]:
# Statistik umum untuk metric cards di tab EDA
eda_summary = {
    'total_observasi': int(len(df)),
    'tahun_min': int(df['year'].min()),
    'tahun_max': int(df['year'].max()),
    'rata_odometer': float(df['odometer'].mean()),
    'merek_dominan': df['manufacturer'].value_counts().idxmax(),
}
with open(f'{OUTPUT_DIR}/eda_summary.json', 'w') as f:
    json.dump(eda_summary, f, indent=2)
print(eda_summary)

# Rata-rata harga per merek (asli)
harga_per_merek = df.groupby('manufacturer')['price'].mean().sort_values(ascending=False)
harga_per_merek.to_csv(f'{OUTPUT_DIR}/harga_per_merek.csv', header=['avg_price'])
print("\nRata-rata harga per merek:")
print(harga_per_merek)

# Rata-rata harga per kondisi (asli)
harga_per_kondisi = df.groupby('condition')['price'].mean().reindex(conditions_app)
harga_per_kondisi.to_csv(f'{OUTPUT_DIR}/harga_per_kondisi.csv', header=['avg_price'])
print("\nRata-rata harga per kondisi:")
print(harga_per_kondisi)

# Data actual vs predicted (test set) untuk plot kalibrasi & residual -> ASLI, bukan np.random
df_actual_vs_pred = pd.DataFrame({
    'actual': y_test.values,
    'predicted': pred_xgb if nama_model_terbaik == 'XGBoost' else model_final.predict(X_test)
})
df_actual_vs_pred['residual'] = df_actual_vs_pred['actual'] - df_actual_vs_pred['predicted']
df_actual_vs_pred.to_csv(f'{OUTPUT_DIR}/actual_vs_predicted.csv', index=False)
df_actual_vs_pred.head()

{'total_observasi': 125271, 'tahun_min': 2000, 'tahun_max': 2022, 'rata_odometer': 102218.81622242977, 'merek_dominan': 'ford'}

Rata-rata harga per merek:
manufacturer
ford      21801.367171
toyota    17860.718877
nissan    13462.412578
honda     12009.312081
Name: price, dtype: float64

Rata-rata harga per kondisi:
condition
excellent    15291.849568
good         20087.924769
fair          4110.665046
poor                  NaN
Name: price, dtype: float64


,actual,predicted,residual
0,22995,22830.291016,164.708984
1,16590,16560.980469,29.019531
2,21519,17022.789062,4496.210938
3,6499,8524.087891,-2025.087891
4,29000,37143.136719,-8143.136719
